# Calibração de multiplicadores

In [1]:
import pandas as pd
import os

In [2]:
working_dir = os.getcwd()
path_dados = os.path.join(working_dir, "..", "data/processed")

path = os.path.join(path_dados, r"bank_marketing_processed.parquet")
df = pd.read_parquet(path)
print(df.shape)
df.head()

(41176, 20)


,age,job,marital,education,default,housing,loan,contact,month,day_of_week,campaign,previous,poutcome,cons.price.idx,cons.conf.idx,euribor3m,y,contacted_before,had_previous_contact,age_group
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,1,0,nonexistent,93.994,-36.4,4.857,no,0,0,50-60
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,1,0,nonexistent,93.994,-36.4,4.857,no,0,0,50-60
2,37,services,married,high.school,no,yes,no,telephone,may,mon,1,0,nonexistent,93.994,-36.4,4.857,no,0,0,30-40
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,1,0,nonexistent,93.994,-36.4,4.857,no,0,0,30-40
4,56,services,married,high.school,no,no,yes,telephone,may,mon,1,0,nonexistent,93.994,-36.4,4.857,no,0,0,50-60


## Definindo a base de conversão

In [3]:
# Conversão para o depósito a prazo
base_rate = (df['y'] == 'yes').mean()
base_rate

np.float64(0.11266271614532737)

### Criação de Segmentos - Baseado nas análises do notebook *notebooks/01_eda.ipynb*

**Variáveis com o maior poder discriminador:**
- poutcome;
- month (mar/sep/oct/dez);
- job (student/retired/admin);
- contact (cellular);
- education (university.degree. *illiterate* tem um volume baixo e pode ser ruído);

In [4]:
# Segmento 1: previous_converter
df['previous_converter'] = df['poutcome'] == 'success'

# Segmento 2: student + cellular 
df['student_digital'] = (df['job'] == 'student') & (df['contact'] == 'cellular')

# Segmento 3: retired
df['retired'] = df['job'] == 'retired'

# Segmento 4: high_season — meses mar, sep, oct, dec
df['high_season'] = df['month'].isin(['mar', 'sep', 'oct', 'dec'])

In [5]:
# Converter y para binário primeiro
df['y_bin'] = (df['y'] == 'yes').astype(int)

# Base rate
base = df['y_bin'].mean()
for seg in ['previous_converter', 'student_digital', 'retired', 'high_season']:
    rate = df[df[seg]]['y_bin'].mean()
    print(f"{seg}: {rate:.3f} → multiplier {rate/base:.2f}x")

previous_converter: 0.651 → multiplier 5.78x
student_digital: 0.364 → multiplier 3.23x
retired: 0.253 → multiplier 2.24x
high_season: 0.465 → multiplier 4.12x
